In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load the dataset
df = pd.read_excel("Sequence_Spoilage_Data.xlsx")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9025 entries, 0 to 9024
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Column1              9025 non-null   int64         
 1   SEQ_NBR              9025 non-null   int64         
 2   SEQ_SCHD_START_DT    9025 non-null   datetime64[ns]
 3   FLEET                9025 non-null   int64         
 4   BASE                 9025 non-null   object        
 5   DIVISION             9025 non-null   object        
 6   SPOILAGE             9025 non-null   object        
 7   TOTAL_BLOCKED_HRS    4420 non-null   float64       
 8   TOTAL_SPOILED_HRS    2719 non-null   float64       
 9   SEQ_CAL_DAYS         9025 non-null   int64         
 10  SEQ_DUTY_DAYS        9025 non-null   int64         
 11  SEQ_TTL_FLTTIME      9025 non-null   int64         
 12  MIN_FLYTIME_PER_LEG  9025 non-null   int64         
 13  MAX_LEGS_PER_DAY     9025 non-nul

In [25]:
# Data cleaning
df.fillna(0, inplace=True)
df.drop_duplicates(inplace=True)
# Modify the dataset to calculate the estimated arrival time
df["SEQ_START_CLEAN"] = df["SEQ_START"].str.replace("Starts", "", regex=False)
df["SEQ_START_DT"] = df["SEQ_SCHD_START_DT"] + pd.to_timedelta(df["SEQ_START_CLEAN"] + ":00")
df["ARRIVAL_ESTIMATE"] = (df["SEQ_START_DT"] + pd.to_timedelta(df["TOTAL_BLOCKED_HRS"], unit="h")+ pd.to_timedelta(df["LAYOVER"], unit="h"))
# Find the actual arrival time
df["FINAL_ARRIVAL"] = (df["SEQ_START_DT"] + pd.to_timedelta(df["TOTAL_BLOCKED_HRS"], unit="h")+ pd.to_timedelta(df["LAYOVER"], unit="h") + pd.to_timedelta(df["TOTAL_SPOILED_HRS"], unit="h"))


Identify where the variables for calculation live:

| Variable | Column Nmae | 
|---|---|
| Fleet code | FLEET |
| Departure | BASE |
| Takeoff time | SEQ_START_CLEAN | 
| Arrival estimate| ARRIVAL_ESTIMATE | 
| Final arrival time| FINAL_ARRIVAL |



In [33]:
# Create a joined dataset for the variables above
joined_df = df.loc[:, ['FLEET', 'BASE', 'SEQ_START_CLEAN','ARRIVAL_ESTIMATE', 'FINAL_ARRIVAL']]
joined_df.head(20)


,FLEET,BASE,SEQ_START_CLEAN,ARRIVAL_ESTIMATE,FINAL_ARRIVAL
0,320,CLT,10:25,2025-02-06 22:25:00,2025-02-06 22:25:00.000000000
1,777,DFW,15:15,2024-04-19 11:40:12,2024-04-19 11:40:12.000000000
2,737,LAX,7:45,2025-05-11 22:45:00,2025-05-11 22:45:00.000000000
3,737,DFW,10:0,2024-12-31 11:00:00,2024-12-31 11:00:00.000000000
4,737,MIA,11:11,2024-07-06 17:41:00,2024-07-06 17:41:00.000000000
5,737,MIA,10:1,2024-10-01 10:01:00,2024-10-01 10:01:00.000000000
6,320,DFW,17:15,2024-09-17 15:36:36,2024-09-17 18:50:24.000000000
7,320,MIA,16:3,2025-04-10 07:03:00,2025-04-10 07:03:00.000000000
8,737,MIA,9:15,2024-04-07 14:48:36,2024-04-07 20:23:23.999999999
9,320,DFW,7:14,2024-11-07 07:14:00,2024-11-07 07:14:00.000000000
